In [6]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score
from typing import Tuple, List, Any
from lightgbm import LGBMRegressor

import warnings, os
import mysql.connector as mysql
warnings.filterwarnings('ignore')
username = os.environ['MYSQL_user']
password = os.environ['MYSQL_password']
DB = mysql.connect(host = "localhost", user = username, passwd = password, database = "AIRBNB")
cursor = DB.cursor(buffered=True)
SEED = 17
compare_metric_name = 'RMSE'

## Utils

In [7]:
def read_table_from_db(table_name):
    df = pd.read_sql(f'SELECT * FROM {table_name}', con=DB)
    for col in df.columns:
        if len(df[col].unique()) == 2 or (df[col].dtype == 'object' and len(df[col].unique()) < 10):
            df[col] = df[col].astype('category')
    return df

In [8]:
def perform_cv(X: pd.DataFrame, y: pd.Series, algorithm: Any, cv: sklearn.model_selection = KFold(n_splits=5, shuffle=True, random_state=SEED), metric: sklearn.metrics = root_mean_squared_error) -> Tuple[List[float], List[float]]:
    """
    Perform cross-validation and return list of scores
    
    Args:
        X (pd.DataFrame): input data
        y (pd.Series): target data
        algorithm (Any): algorithm to use for training and prediction
        cv (sklearn.model_selection, default=KFold(n_splits=5, shuffle=True, random_state=SEED)): cross-validation strategy
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[List[float], List[float]]: Tuple of lists of train and validation scores
    """
    train_scores, validation_scores = [], []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        algorithm.fit(X_train, y_train)
        y_train_pred = algorithm.predict(X_train)
        y_val_pred = algorithm.predict(X_val)
        train_scores.append(metric(y_train, y_train_pred))
        validation_scores.append(metric(y_val, y_val_pred))
    return train_scores, validation_scores

def evaluation(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series, algorithm: Any, metric: sklearn.metrics = root_mean_squared_error) -> Tuple[float, float, np.ndarray]:
    """
    Train the algorithm on the train data and evaluate on the train and test data
    
    Args:
        X_train (pd.DataFrame): input train data
        y_train (pd.Series): target train data
        X_test (pd.DataFrame): input test data
        y_test (pd.Series): target test data
        algorithm (Any): algorithm to use for training and prediction
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[float, float, np.ndarray]: train_score, test_score, predictions on test data
    """
    algorithm.fit(X_train, y_train)
    y_train_pred = algorithm.predict(X_train)
    y_test_pred = algorithm.predict(X_test)
    train_results = metric(y_train, y_train_pred)
    test_results = metric(y_test, y_test_pred)
    return train_results, test_results, y_test_pred

# Load dataset

In [9]:
data = read_table_from_db('airbnb_data')
target_feature = 'log_price'
radius_meters = 100
location_features = ['distance_to_nearest_crime_m', f'number_of_crimes_within_{radius_meters}m', f'average_offence_weight_within_{radius_meters}m',
                    'distance_to_nearest_bus_stop_m', f'number_of_bus_stops_within_{radius_meters}m',
                    'distance_to_nearest_subway_station_m', f'number_of_subway_stations_within_{radius_meters}m',
                    'distance_to_nearest_restaurant_m', f'number_of_restaurants_within_{radius_meters}m',
                    'distance_to_nearest_education_institution_m', f'number_of_education_institutions_within_{radius_meters}m',
                    'distance_to_nearest_cultural_institution_m', f'number_of_cultural_institutions_within_{radius_meters}m',
                    'distance_to_nearest_recreation_point_m', f'number_of_recreation_points_within_{radius_meters}m',
                    'distance_to_nearest_religious_institution_m', f'number_of_religious_institutions_within_{radius_meters}m',
                    'distance_to_nearest_health_institution_m', f'number_of_health_institutions_within_{radius_meters}m',
                    'distance_to_nearest_main_attraction_m']
data = data[location_features + [target_feature]]
data

,distance_to_nearest_crime_m,number_of_crimes_within_100m,average_offence_weight_within_100m,distance_to_nearest_bus_stop_m,number_of_bus_stops_within_100m,distance_to_nearest_subway_station_m,number_of_subway_stations_within_100m,distance_to_nearest_restaurant_m,number_of_restaurants_within_100m,distance_to_nearest_education_institution_m,...,distance_to_nearest_cultural_institution_m,number_of_cultural_institutions_within_100m,distance_to_nearest_recreation_point_m,number_of_recreation_points_within_100m,distance_to_nearest_religious_institution_m,number_of_religious_institutions_within_100m,distance_to_nearest_health_institution_m,number_of_health_institutions_within_100m,distance_to_nearest_main_attraction_m,log_price
0,44.50410,65,135.7380,73.7565,1,113.163,0,29.4473,18,290.1910,...,264.899,0,74.9974,1,113.3110,0,748.997,0,497.315,5.48064
1,45.13680,29,97.1724,51.7841,3,467.638,0,72.7958,2,96.2377,...,114.917,0,157.0960,0,143.3380,0,798.512,0,5687.510,4.27667
2,17.42260,59,112.7970,93.1094,1,277.759,0,43.5074,1,61.5241,...,441.078,0,122.0370,0,489.0130,0,867.072,0,5022.830,4.39445
3,46.58130,42,104.4760,101.8910,0,284.301,0,52.1928,3,138.6770,...,571.057,0,108.9370,0,157.2840,0,262.918,0,2800.150,4.17439
4,11.24030,77,121.5970,30.0502,1,315.957,0,40.2882,3,265.2790,...,404.074,0,67.3330,2,262.1370,0,436.210,0,1609.260,4.17439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19454,16.11150,33,194.0610,54.3359,2,358.649,0,190.6080,0,74.9113,...,715.725,0,221.5280,0,432.0350,0,1871.660,0,9160.790,4.06044
19455,22.24270,6,108.8330,233.6070,0,195.015,0,184.7880,0,154.5870,...,207.671,0,164.9140,0,23.2916,1,1198.130,0,8735.350,4.33073
19456,62.64900,13,112.3080,206.1970,0,512.000,0,172.5360,0,15.6187,...,314.328,0,374.1010,0,263.4550,0,309.404,0,6281.460,3.71357
19457,4.68686,70,105.1430,34.8370,1,322.696,0,25.6413,4,97.1276,...,512.488,0,128.8660,0,445.6860,0,952.257,0,4996.590,4.75359


## Split dataset

In [10]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=SEED)

## Base model

In [11]:
model = LGBMRegressor(random_state=SEED, verbose=-1, linear_tree=True)
train_scores, validation_scores = perform_cv(train_data[location_features], train_data[target_feature], model, cv=KFold(n_splits=5, shuffle=True, random_state=SEED), metric=root_mean_squared_error)
print(f"Train {compare_metric_name}: {np.mean(train_scores):.4f} +- {np.std(train_scores):.4f}")
print(f"Validation {compare_metric_name}: {np.mean(validation_scores):.4f} +- {np.std(validation_scores):.4f}")

Train RMSE: 0.4844 +- 0.0021
Validation RMSE: 0.6189 +- 0.0083


In [13]:
X_train = train_data[location_features]
y_train = train_data[target_feature]
X_test = test_data[location_features]
y_test = test_data[target_feature]
model = LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1, objective='regression', metric=compare_metric_name, linear_tree=True)
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
metrics = {
    'R2': r2_score,
    'R2_adj': lambda y_true, y_pred: 1 - (1 - r2_score(y_true, y_pred)) * (len(y_true) - 1) / (len(y_true) - X_train.shape[1] - 1),
    'MSE': mean_squared_error,
    'RMSE': root_mean_squared_error,
    'MAE': mean_absolute_error,
}
results = {}
for metric_name, metric in metrics.items():
    train_score = metric(y_train, y_train_pred)
    test_score = metric(y_test, y_test_pred)
    results[metric_name] = {
        'train': train_score,
        'test': test_score
    }
df = pd.DataFrame(results).T
df

,train,test
R2,0.530737,0.332674
R2_adj,0.530133,0.329226
MSE,0.254877,0.376545
RMSE,0.504853,0.613633
MAE,0.400915,0.491524
